# Stream Orders to Bronze

Reads order events from `evh_brazilian_ecommerce` via Spark Structured Streaming (Kafka-compatible endpoint) and writes them into `dbr_dev.brazilian_ecommerce_bronze.brz_orders`.

The native Event Hubs Spark connector doesn't reliably work on our shared cluster, so this uses Event Hub's Kafka-compatible surface instead — no extra library install needed, works on any cluster.

`discount_code` is declared **nullable in the schema from the start**, so the pipeline can handle it appearing in events later without any restart — see `03_schema_evolution_demo` for that walkthrough.

## 1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import re

## 2. Shared config
Pulls in `bronze_schema` and friends from the shared utilities notebook, kept in sync with the batch ingestion side.

In [0]:
%run /Workspace/Users/yanquiel@softserve.academy/ecommerce-bronze-platform-/notebooks/utilities

## 3. Parameters

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("eventhub_name", "evh_brazilian_ecommerce", "Event Hub Name")

catalog = dbutils.widgets.get("catalog")
eventhub_name = dbutils.widgets.get("eventhub_name")

## 4. Event Hub connection
Connection string comes from the governed, Key Vault-backed secret scope — never hardcoded, never printed.

In [0]:
connection_string = dbutils.secrets.get(
    scope="e-commerce-bronze-scope",
    key="evh-brazilian-ecommerce"
)

## 5. Kafka connection options
Event Hub exposes a Kafka-compatible endpoint on port 9093, authenticated via SASL_SSL / PLAIN with the connection string as the password.

**Important:** the JAAS config must use the *shaded* class name (`kafkashaded.org.apache.kafka...`), not the normal one. Databricks Runtime relocates the Kafka client classes internally — using the unshaded name throws a `ClassNotFoundException` deep inside `KafkaAdminClient`, which surfaces as an unhelpful `Failed to create new KafkaAdminClient` error with no obvious cause.

In [0]:
namespace_match = re.search(r"sb://([^./]+)\.servicebus\.windows\.net", connection_string)
namespace = namespace_match.group(1)
bootstrap_servers = f"{namespace}.servicebus.windows.net:9093"

sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_string}";'
)

kafka_options = {
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": sasl_config,
    "subscribe": eventhub_name,
    "startingOffsets": "latest",
    "failOnDataLoss": "false",  # Event Hub retention can outpace checkpoint restarts; don't hard-fail the stream over it
}

## 6. Read the stream

In [0]:
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("kafka.security.protocol", "SASL_SSL")
    .options(**kafka_options)
    .load()
)

## 7. Order schema
`discount_code` is included here as nullable even though it isn't in every event yet — that's the whole mechanism behind the schema-evolution demo. `from_json` returns `null` for a declared field that's missing from a given payload, and the real value once it's present, with no restart needed either way.

In [0]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("discount_code", StringType(), True),
])

## 8. Parse JSON + add metadata columns
`source` and `ingestion_timestamp` are the governed metadata columns required across all bronze tables.

In [0]:
parsed_df = (
    raw_df
    .select(
        F.col("value").cast("string").alias("json_payload"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
    )
    .withColumn("data", F.from_json(F.col("json_payload"), order_schema))
    .select("data.*", "kafka_partition", "kafka_offset")
    .withColumn("source", F.lit(eventhub_name))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

## 9. Write to bronze
Checkpoint lives in the dedicated `checkpoints` volume (sibling to `landing`, not nested under it), matching the current infra setup. Deleting this checkpoint and restarting is the safe way to fully reload from scratch if ever needed.

In [0]:
checkpoint_path = f"/Volumes/{catalog}/{bronze_schema}/landing/checkpoints/brz_orders"

query = (
    parsed_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{bronze_schema}.brz_orders")
)

## 10. Monitor the stream
Check these while the producer is running to confirm events are actually flowing, and during the schema-evolution demo to confirm the stream never restarts.

In [0]:
query.status

In [0]:
query.lastProgress

## 11. Stop the stream
Run when done — leaving a streaming query running unattended burns cluster time for nothing.

In [0]:
query.stop()